In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from clv_dataset import build_retention_dataset
from preprocessing import preprocess_online_retail


# ============================================================
# OUTPUT FOLDERS
# ============================================================

OUTPUT_DIR = "clustering_outputs"
PLOT_DIR = "clustering_plots"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)


# ============================================================
# LOAD CLEANED MASTER DATASET
# ============================================================

df = preprocess_online_retail(
    file_path="online_retail_II.xlsx",
    verbose=False
)


# ============================================================
# BUILD CUSTOMER RETENTION / CLV DATASET
# ============================================================

clv_df = build_retention_dataset(
    df,
    cutoff_date="2011-09-09",
    prediction_days=90,
    active_days=180,
    verbose=True
)

# Ensure target exists
if "FutureSpend90Days" not in clv_df.columns:
    raise ValueError("FutureSpend90Days column is missing from clv_df.")

# Correct retention label
clv_df["RetentionLabel"] = (clv_df["FutureSpend90Days"] > 0).astype(int)


# ============================================================
# CORRECT DATASET SUMMARY
# ============================================================

total_customers = len(clv_df)
future_positive = (clv_df["FutureSpend90Days"] > 0).sum()
future_zero = (clv_df["FutureSpend90Days"] == 0).sum()
avg_future_spend = clv_df["FutureSpend90Days"].mean()

print("\nCorrect Clustering Dataset Summary:")
print(f"Total customers: {total_customers}")
print(f"Customers with future spend > 0: {future_positive}")
print(f"Customers with future spend = 0: {future_zero}")
print(f"Average future spend: {avg_future_spend:.2f}")


# ============================================================
# HANDLE AvgGapDays PROPERLY
# ============================================================

if "AvgGapDays" in clv_df.columns:
    clv_df["AvgGapDaysMissing"] = clv_df["AvgGapDays"].isna().astype(int)
else:
    raise ValueError("AvgGapDays column is missing from clv_df.")


# ============================================================
# DEFINE CLUSTERING FEATURES
# IMPORTANT:
# Do NOT include FutureSpend90Days or RetentionLabel in clustering.
# They are only used later for profiling.
# ============================================================

clustering_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AvgBasketValue",
    "AvgQuantity",
    "UniqueProducts",
    "LifetimeDays",
    "PurchaseRate",
    "AvgGapDays",
    "AvgGapDaysMissing",
    "StdGapDays",
    "PurchasesLast30Days",
    "SpendLast30Days",
    "PurchasesLast90Days",
    "SpendLast90Days",
    "ReturnRate",
    "RevenuePerDay",
    "AvgSpendPerProduct",
    "ProductDiversityRate",
    "RecencyFrequency",
    "SpendTrendRatio",
    "FrequencyLast90DaysRatio",
    "IsNewCustomer"
]

missing_features = [col for col in clustering_features if col not in clv_df.columns]

if missing_features:
    raise ValueError(f"Missing clustering features: {missing_features}")

X_cluster = clv_df[clustering_features].copy()

print("\nClustering features used:")
print(clustering_features)


# ============================================================
# FEATURE GROUPS
# ============================================================

log_features = [
    "Frequency",
    "Monetary",
    "AvgBasketValue",
    "UniqueProducts",
    "SpendLast30Days",
    "SpendLast90Days",
    "RevenuePerDay",
    "AvgSpendPerProduct",
    "ProductDiversityRate",
    "RecencyFrequency"
]

scale_features = [
    "Recency",
    "AvgQuantity",
    "LifetimeDays",
    "PurchaseRate",
    "AvgGapDays",
    "AvgGapDaysMissing",
    "StdGapDays",
    "PurchasesLast30Days",
    "PurchasesLast90Days",
    "ReturnRate",
    "SpendTrendRatio",
    "FrequencyLast90DaysRatio",
    "IsNewCustomer"
]

# Keep only columns that actually exist
log_features = [col for col in log_features if col in X_cluster.columns]
scale_features = [col for col in scale_features if col in X_cluster.columns]


# ============================================================
# SAFE LOG TRANSFORM
# Clips negative values to zero before log1p.
# This avoids invalid values if returns create negative spend.
# ============================================================

def safe_log1p(x):
    return np.log1p(np.clip(x, a_min=0, a_max=None))


log_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(safe_log1p, validate=False)),
    ("scaler", StandardScaler())
])

scale_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("log", log_pipeline, log_features),
    ("scale", scale_pipeline, scale_features)
], remainder="drop")


X_scaled = preprocessor.fit_transform(X_cluster)

print("\nScaled clustering matrix shape:", X_scaled.shape)


# ============================================================
# PCA FOR VISUALIZATION ONLY
# ============================================================

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "Customer_ID": clv_df["Customer_ID"].values,
    "PCA1": pca_coords[:, 0],
    "PCA2": pca_coords[:, 1]
})

explained_variance = pca.explained_variance_ratio_.sum()

print(f"\nPCA explained variance using 2 components: {explained_variance:.4f}")


# ============================================================
# KMEANS K SEARCH
# ============================================================

print("\nRunning KMeans k-search...")

kmeans_results = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    ch_score = calinski_harabasz_score(X_scaled, labels)

    kmeans_results.append({
        "K": k,
        "Inertia": inertia,
        "Silhouette_Score": silhouette,
        "Davies_Bouldin_Score": db_score,
        "Calinski_Harabasz_Score": ch_score
    })

kmeans_results_df = pd.DataFrame(kmeans_results)

print("\nKMeans k-search results:")
print(kmeans_results_df)

kmeans_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "kmeans_k_search_results.csv"),
    index=False
)


# ============================================================
# DBSCAN PARAMETER SEARCH
# Diagnostic only.
# DBSCAN is rejected later if noise is too high.
# ============================================================

print("\nRunning DBSCAN parameter search...")

dbscan_results = []

eps_values = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
min_samples_values = [5, 10, 15, 20]

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        labels = dbscan.fit_predict(X_scaled)

        unique_labels = set(labels)
        clusters = len(unique_labels - {-1})
        noise_count = np.sum(labels == -1)
        noise_pct = noise_count / len(labels) * 100

        # Only score when at least 2 real clusters exist
        if clusters >= 2:
            valid_mask = labels != -1

            # Score only non-noise points
            if valid_mask.sum() > 1 and len(set(labels[valid_mask])) >= 2:
                silhouette = silhouette_score(X_scaled[valid_mask], labels[valid_mask])
                db_score = davies_bouldin_score(X_scaled[valid_mask], labels[valid_mask])
                ch_score = calinski_harabasz_score(X_scaled[valid_mask], labels[valid_mask])
            else:
                silhouette = np.nan
                db_score = np.nan
                ch_score = np.nan
        else:
            silhouette = np.nan
            db_score = np.nan
            ch_score = np.nan

        dbscan_results.append({
            "eps": eps,
            "min_samples": min_samples,
            "Clusters": clusters,
            "Noise_Count": noise_count,
            "Noise_%": noise_pct,
            "Silhouette_Score": silhouette,
            "Davies_Bouldin_Score": db_score,
            "Calinski_Harabasz_Score": ch_score
        })

dbscan_results_df = pd.DataFrame(dbscan_results)

dbscan_results_df = dbscan_results_df.sort_values(
    by=["Noise_%", "Silhouette_Score"],
    ascending=[True, False]
)

print("\nDBSCAN search results:")
print(dbscan_results_df.head(15))

dbscan_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "dbscan_parameter_search_results.csv"),
    index=False
)


# ============================================================
# FINAL MODEL CHOICE
# K=4 is chosen for business interpretability.
# K=2 may have better pure clustering metrics, but K=4 gives
# more useful customer segments.
# ============================================================

FINAL_K = 4


# ============================================================
# TRAIN FINAL CLUSTERING MODELS
# ============================================================

models = {
    "KMeans": KMeans(
        n_clusters=FINAL_K,
        random_state=42,
        n_init=20
    ),
    "Agglomerative": AgglomerativeClustering(
        n_clusters=FINAL_K
    ),
    "GaussianMixture": GaussianMixture(
        n_components=FINAL_K,
        random_state=42
    )
}

model_results = {}

for model_name, model in models.items():
    print(f"\nRunning {model_name}...")

    if model_name == "GaussianMixture":
        labels = model.fit_predict(X_scaled)
    else:
        labels = model.fit_predict(X_scaled)

    clusters = len(set(labels))
    noise_count = np.sum(labels == -1)
    noise_pct = noise_count / len(labels) * 100

    silhouette = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    ch_score = calinski_harabasz_score(X_scaled, labels)

    model_results[model_name] = {
        "labels": labels,
        "Clusters": clusters,
        "Noise_Count": noise_count,
        "Noise_%": noise_pct,
        "Silhouette_Score": silhouette,
        "Davies_Bouldin_Score": db_score,
        "Calinski_Harabasz_Score": ch_score
    }

    print(f"Clusters found: {clusters}")
    print(f"Noise count: {noise_count}")
    print(f"Noise %: {noise_pct:.2f}")
    print(f"Silhouette Score: {silhouette:.4f}")
    print(f"Davies Bouldin Score: {db_score:.4f}")
    print(f"Calinski Harabasz Score: {ch_score:.4f}")


# ============================================================
# OPTIONAL DBSCAN FINAL DIAGNOSTIC
# Pick best DBSCAN with acceptable noise if possible.
# Reject if noise is too high.
# ============================================================

acceptable_dbscan = dbscan_results_df[
    (dbscan_results_df["Clusters"] >= 2) &
    (dbscan_results_df["Noise_%"] <= 40)
].copy()

if len(acceptable_dbscan) > 0:
    best_dbscan_row = acceptable_dbscan.sort_values(
        by="Silhouette_Score",
        ascending=False
    ).iloc[0]
else:
    best_dbscan_row = dbscan_results_df.iloc[0]

best_eps = best_dbscan_row["eps"]
best_min_samples = int(best_dbscan_row["min_samples"])

print("\nRunning DBSCAN diagnostic...")
print(f"Best DBSCAN eps: {best_eps}")
print(f"Best DBSCAN min_samples: {best_min_samples}")

dbscan_final = DBSCAN(
    eps=best_eps,
    min_samples=best_min_samples
)

dbscan_labels = dbscan_final.fit_predict(X_scaled)

dbscan_clusters = len(set(dbscan_labels) - {-1})
dbscan_noise_count = np.sum(dbscan_labels == -1)
dbscan_noise_pct = dbscan_noise_count / len(dbscan_labels) * 100

if dbscan_clusters >= 2:
    valid_mask = dbscan_labels != -1

    if valid_mask.sum() > 1 and len(set(dbscan_labels[valid_mask])) >= 2:
        dbscan_silhouette = silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
        dbscan_db_score = davies_bouldin_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
        dbscan_ch_score = calinski_harabasz_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
    else:
        dbscan_silhouette = np.nan
        dbscan_db_score = np.nan
        dbscan_ch_score = np.nan
else:
    dbscan_silhouette = np.nan
    dbscan_db_score = np.nan
    dbscan_ch_score = np.nan

model_results["DBSCAN"] = {
    "labels": dbscan_labels,
    "Clusters": dbscan_clusters,
    "Noise_Count": dbscan_noise_count,
    "Noise_%": dbscan_noise_pct,
    "Silhouette_Score": dbscan_silhouette,
    "Davies_Bouldin_Score": dbscan_db_score,
    "Calinski_Harabasz_Score": dbscan_ch_score
}

print(f"DBSCAN clusters found: {dbscan_clusters}")
print(f"DBSCAN noise count: {dbscan_noise_count}")
print(f"DBSCAN noise %: {dbscan_noise_pct:.2f}")


# ============================================================
# MODEL COMPARISON
# ============================================================

comparison_rows = []

for model_name, result in model_results.items():
    comparison_rows.append({
        "Model": model_name,
        "Clusters": result["Clusters"],
        "Noise_Count": result["Noise_Count"],
        "Noise_%": result["Noise_%"],
        "Silhouette_Score": result["Silhouette_Score"],
        "Davies_Bouldin_Score": result["Davies_Bouldin_Score"],
        "Calinski_Harabasz_Score": result["Calinski_Harabasz_Score"]
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.sort_values(
    by="Silhouette_Score",
    ascending=False
)

print("\nFinal Clustering Comparison:")
print(comparison_df)

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "clustering_model_comparison.csv"),
    index=False
)


# ============================================================
# ADD CLUSTERS TO DATASET
# ============================================================

clustered_customers = clv_df.copy()

clustered_customers["KMeans_Cluster"] = model_results["KMeans"]["labels"]
clustered_customers["Agglomerative_Cluster"] = model_results["Agglomerative"]["labels"]
clustered_customers["GaussianMixture_Cluster"] = model_results["GaussianMixture"]["labels"]
clustered_customers["DBSCAN_Cluster"] = model_results["DBSCAN"]["labels"]

pca_df["KMeans_Cluster"] = clustered_customers["KMeans_Cluster"].values
pca_df["Agglomerative_Cluster"] = clustered_customers["Agglomerative_Cluster"].values
pca_df["GaussianMixture_Cluster"] = clustered_customers["GaussianMixture_Cluster"].values
pca_df["DBSCAN_Cluster"] = clustered_customers["DBSCAN_Cluster"].values


# ============================================================
# CLUSTER COUNTS
# ============================================================

print("\nKMeans Cluster Counts:")
print(clustered_customers["KMeans_Cluster"].value_counts().sort_index())

print("\nAgglomerative Cluster Counts:")
print(clustered_customers["Agglomerative_Cluster"].value_counts().sort_index())

print("\nGaussianMixture Cluster Counts:")
print(clustered_customers["GaussianMixture_Cluster"].value_counts().sort_index())

print("\nDBSCAN Cluster Counts:")
print(clustered_customers["DBSCAN_Cluster"].value_counts().sort_index())


# ============================================================
# KMEANS CLUSTER PROFILE
# ============================================================

profile_rows = []

for cluster_id in sorted(clustered_customers["KMeans_Cluster"].unique()):
    temp = clustered_customers[clustered_customers["KMeans_Cluster"] == cluster_id]

    cluster_size = len(temp)
    total_future_revenue = temp["FutureSpend90Days"].sum()
    total_historical_revenue = temp["Monetary"].sum()
    retained_customers = (temp["FutureSpend90Days"] > 0).sum()

    profile_rows.append({
        "KMeans_Cluster": cluster_id,
        "ClusterSize": cluster_size,
        "CustomerShare_%": cluster_size / len(clustered_customers) * 100,

        "AvgRecency": temp["Recency"].mean(),
        "AvgFrequency": temp["Frequency"].mean(),
        "AvgMonetary": temp["Monetary"].mean(),
        "TotalHistoricalRevenue": total_historical_revenue,

        "AvgBasketValue": temp["AvgBasketValue"].mean(),
        "AvgQuantity": temp["AvgQuantity"].mean(),
        "UniqueProducts_mean": temp["UniqueProducts"].mean(),
        "LifetimeDays_mean": temp["LifetimeDays"].mean(),
        "PurchaseRate_mean": temp["PurchaseRate"].mean(),
        "AvgGapDays": temp["AvgGapDays"].mean(),
        "AvgGapDaysMissingRate_%": temp["AvgGapDaysMissing"].mean() * 100,
        "StdGapDays_mean": temp["StdGapDays"].mean(),

        "PurchasesLast30Days_mean": temp["PurchasesLast30Days"].mean(),
        "SpendLast30Days_mean": temp["SpendLast30Days"].mean(),
        "PurchasesLast90Days_mean": temp["PurchasesLast90Days"].mean(),
        "SpendLast90Days_mean": temp["SpendLast90Days"].mean(),

        "ReturnRate_mean": temp["ReturnRate"].mean(),
        "RevenuePerDay_mean": temp["RevenuePerDay"].mean(),
        "AvgSpendPerProduct_mean": temp["AvgSpendPerProduct"].mean(),
        "ProductDiversityRate_mean": temp["ProductDiversityRate"].mean(),
        "RecencyFrequency_mean": temp["RecencyFrequency"].mean(),
        "SpendTrendRatio_mean": temp["SpendTrendRatio"].mean(),
        "FrequencyLast90DaysRatio_mean": temp["FrequencyLast90DaysRatio"].mean(),
        "IsNewCustomer_mean": temp["IsNewCustomer"].mean(),

        "AvgFutureSpend90Days": temp["FutureSpend90Days"].mean(),
        "TotalFutureRevenue": total_future_revenue,
        "FutureRevenueShare_%": total_future_revenue / clustered_customers["FutureSpend90Days"].sum() * 100,
        "RetentionRate_%": retained_customers / cluster_size * 100,
        "ZeroFutureSpendRate_%": (temp["FutureSpend90Days"] == 0).mean() * 100
    })

profile_df = pd.DataFrame(profile_rows)


# ============================================================
# AUTOMATIC SEGMENT NAMING BASED ON PROFILE
# ============================================================

# Rank clusters by future revenue, retention, recency, and frequency
profile_df["RevenueRank"] = profile_df["AvgFutureSpend90Days"].rank(ascending=False)
profile_df["RetentionRank"] = profile_df["RetentionRate_%"].rank(ascending=False)
profile_df["RecencyRank"] = profile_df["AvgRecency"].rank(ascending=True)
profile_df["FrequencyRank"] = profile_df["AvgFrequency"].rank(ascending=False)

segment_names = {}

best_cluster = profile_df.sort_values(
    by=["AvgFutureSpend90Days", "RetentionRate_%", "AvgFrequency"],
    ascending=[False, False, False]
).iloc[0]["KMeans_Cluster"]

new_cluster = profile_df.sort_values(
    by=["IsNewCustomer_mean", "AvgFrequency"],
    ascending=[False, True]
).iloc[0]["KMeans_Cluster"]

inactive_cluster = profile_df.sort_values(
    by=["AvgRecency", "ZeroFutureSpendRate_%"],
    ascending=[False, False]
).iloc[0]["KMeans_Cluster"]

for _, row in profile_df.iterrows():
    cluster_id = row["KMeans_Cluster"]

    if cluster_id == best_cluster:
        segment_names[cluster_id] = "High-Value Loyalists"
    elif cluster_id == new_cluster:
        segment_names[cluster_id] = "New / One-Time Customers"
    elif cluster_id == inactive_cluster:
        segment_names[cluster_id] = "At-Risk Inactive Customers"
    else:
        segment_names[cluster_id] = "Regular Mid-Value Customers"

profile_df["SegmentName"] = profile_df["KMeans_Cluster"].map(segment_names)

clustered_customers["KMeans_SegmentName"] = clustered_customers["KMeans_Cluster"].map(segment_names)


# Put useful columns first
front_cols = [
    "Customer_ID",
    "KMeans_Cluster",
    "KMeans_SegmentName",
    "Agglomerative_Cluster",
    "GaussianMixture_Cluster",
    "DBSCAN_Cluster",
    "FutureSpend90Days",
    "RetentionLabel"
]

other_cols = [col for col in clustered_customers.columns if col not in front_cols]
clustered_customers = clustered_customers[front_cols + other_cols]


print("\nFinal KMeans Cluster Profile:")
print(profile_df.sort_values(by="AvgFutureSpend90Days", ascending=False))


# ============================================================
# SAVE OUTPUT FILES
# ============================================================

profile_df.to_csv(
    os.path.join(OUTPUT_DIR, "kmeans_cluster_profiles.csv"),
    index=False
)

clustered_customers.to_csv(
    os.path.join(OUTPUT_DIR, "clustered_customers.csv"),
    index=False
)

pca_df.to_csv(
    os.path.join(OUTPUT_DIR, "pca_cluster_coordinates.csv"),
    index=False
)

feature_means = clustered_customers.groupby("KMeans_Cluster")[clustering_features].mean()

feature_means.to_csv(
    os.path.join(OUTPUT_DIR, "kmeans_cluster_feature_means.csv")
)


# ============================================================
# PLOTS
# ============================================================

plt.figure(figsize=(8, 6))
plt.scatter(
    pca_df["PCA1"],
    pca_df["PCA2"],
    c=pca_df["KMeans_Cluster"],
    alpha=0.7
)
plt.title("KMeans Customer Segments - PCA View")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="KMeans Cluster")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "kmeans_pca_clusters.png"), dpi=300)
plt.close()


plt.figure(figsize=(8, 5))
plt.plot(
    kmeans_results_df["K"],
    kmeans_results_df["Inertia"],
    marker="o"
)
plt.title("KMeans Elbow Curve")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "kmeans_elbow_curve.png"), dpi=300)
plt.close()


plt.figure(figsize=(8, 5))
plt.plot(
    kmeans_results_df["K"],
    kmeans_results_df["Silhouette_Score"],
    marker="o"
)
plt.title("KMeans Silhouette Scores")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "kmeans_silhouette_scores.png"), dpi=300)
plt.close()


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\nSaved clustering output files in:")
print(OUTPUT_DIR)

print("\nSaved clustering plot files in:")
print(PLOT_DIR)

print("\nImportant files created:")
print(os.path.join(OUTPUT_DIR, "clustering_model_comparison.csv"))
print(os.path.join(OUTPUT_DIR, "kmeans_k_search_results.csv"))
print(os.path.join(OUTPUT_DIR, "dbscan_parameter_search_results.csv"))
print(os.path.join(OUTPUT_DIR, "kmeans_cluster_profiles.csv"))
print(os.path.join(OUTPUT_DIR, "clustered_customers.csv"))
print(os.path.join(OUTPUT_DIR, "pca_cluster_coordinates.csv"))
print(os.path.join(OUTPUT_DIR, "kmeans_cluster_feature_means.csv"))

print("\nRecommended final segmentation model for frontend:")
print("KMeans with 4 clusters")

print("\nReason:")
print("KMeans is selected because it gives balanced, interpretable customer groups.")
print("DBSCAN is not selected because it marks too many customers as noise.")
print("GaussianMixture may score slightly better, but its clusters are usually less frontend-friendly.")

print("\nClustered Dataset Sample:")
print(clustered_customers.head())

Cutoff date: 2011-09-09
Prediction end: 2011-12-08
Active customer window: 180 days
Final dataset shape: (2778, 26)
Customers avg spend in the 90 days after cutoff:  899.50
Number of customers who had a spending > 0 in the 90 days after cutoff:  2778
Number of customers who had no spending in the 90 days after cutoff:  1071

Correct Clustering Dataset Summary:
Total customers: 2778
Customers with future spend > 0: 1707
Customers with future spend = 0: 1071
Average future spend: 899.50

Clustering features used:
['Recency', 'Frequency', 'Monetary', 'AvgBasketValue', 'AvgQuantity', 'UniqueProducts', 'LifetimeDays', 'PurchaseRate', 'AvgGapDays', 'AvgGapDaysMissing', 'StdGapDays', 'PurchasesLast30Days', 'SpendLast30Days', 'PurchasesLast90Days', 'SpendLast90Days', 'ReturnRate', 'RevenuePerDay', 'AvgSpendPerProduct', 'ProductDiversityRate', 'RecencyFrequency', 'SpendTrendRatio', 'FrequencyLast90DaysRatio', 'IsNewCustomer']

Scaled clustering matrix shape: (2778, 23)

PCA explained variance u

In [3]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# INPUT FOLDERS FROM YOUR CLUSTERING SCRIPT
OUTPUT_DIR = "clustering_outputs"

# NEW FRONTEND OUTPUT FOLDERS
FRONTEND_PLOT_DIR = "clustering_frontend_plots"
FRONTEND_DATA_DIR = "clustering_frontend_data"

os.makedirs(
    FRONTEND_PLOT_DIR,
    exist_ok=True
)

os.makedirs(
    FRONTEND_DATA_DIR,
    exist_ok=True
)


# FILE PATHS
profile_path = os.path.join(
    OUTPUT_DIR,
    "kmeans_cluster_profiles.csv"
)

clustered_customers_path = os.path.join(
    OUTPUT_DIR,
    "clustered_customers.csv"
)

pca_path = os.path.join(
    OUTPUT_DIR,
    "pca_cluster_coordinates.csv"
)

model_comparison_path = os.path.join(
    OUTPUT_DIR,
    "clustering_model_comparison.csv"
)

kmeans_k_search_path = os.path.join(
    OUTPUT_DIR,
    "kmeans_k_search_results.csv"
)

dbscan_search_path = os.path.join(
    OUTPUT_DIR,
    "dbscan_parameter_search_results.csv"
)


# LOADING FILES
profile_df = pd.read_csv(
    profile_path
)

clustered_customers_df = pd.read_csv(
    clustered_customers_path
)

pca_df = pd.read_csv(
    pca_path
)

model_comparison_df = pd.read_csv(
    model_comparison_path
)

kmeans_k_search_df = pd.read_csv(
    kmeans_k_search_path
)

dbscan_search_df = pd.read_csv(
    dbscan_search_path
)


# BASIC VALIDATION
required_profile_cols = [
    "KMeans_Cluster",
    "SegmentName",
    "ClusterSize",
    "CustomerShare_%",
    "AvgRecency",
    "AvgFrequency",
    "AvgMonetary",
    "AvgFutureSpend90Days",
    "TotalFutureRevenue",
    "FutureRevenueShare_%",
    "RetentionRate_%",
    "ZeroFutureSpendRate_%"
]

missing_profile_cols = [
    col
    for col in required_profile_cols
    if col not in profile_df.columns
]

if missing_profile_cols:

    raise ValueError(
        f"Missing columns in kmeans_cluster_profiles.csv: {missing_profile_cols}"
    )


# SORTING SEGMENTS BY FUTURE VALUE
profile_df = profile_df.sort_values(
    by="AvgFutureSpend90Days",
    ascending=False
).reset_index(drop=True)

profile_df["SegmentDisplay"] = (
    profile_df["SegmentName"] +
    "\nCluster " +
    profile_df["KMeans_Cluster"].astype(str)
)


# RECOMMENDATION MAPPING
def get_segment_recommendation(segment_name):

    if "High-Value Loyalists" in segment_name:

        return "Protect with loyalty rewards, VIP campaigns, early access, and personalized recommendations."

    if "Regular Mid-Value" in segment_name:

        return "Grow using cross-sell offers, bundles, category recommendations, and moderate-value campaigns."

    if "At-Risk Inactive" in segment_name:

        return "Reactivate using win-back discounts, reminder campaigns, and limited-time offers."

    if "New / One-Time" in segment_name:

        return "Nurture using onboarding flows, first-repeat-purchase offers, and low-cost engagement campaigns."

    return "Use personalized campaigns based on recent purchase behavior."


def get_segment_insight(row):

    segment_name = row["SegmentName"]

    if "High-Value Loyalists" in segment_name:

        return "Small but extremely valuable group that drives most future revenue."

    if "Regular Mid-Value" in segment_name:

        return "Largest stable customer group with moderate revenue potential."

    if "At-Risk Inactive" in segment_name:

        return "Previously active customers whose long recency makes them risky."

    if "New / One-Time" in segment_name:

        return "Low-frequency customers with limited history and high zero-spend risk."

    return "Behaviorally distinct customer group."


profile_df["Recommendation"] = profile_df["SegmentName"].apply(
    get_segment_recommendation
)

profile_df["Insight"] = profile_df.apply(
    get_segment_insight,
    axis=1
)


# FRONTEND SEGMENT CARDS DATA
segment_cards_df = profile_df[
    [
        "KMeans_Cluster",
        "SegmentName",
        "ClusterSize",
        "CustomerShare_%",
        "AvgRecency",
        "AvgFrequency",
        "AvgMonetary",
        "AvgFutureSpend90Days",
        "TotalFutureRevenue",
        "FutureRevenueShare_%",
        "RetentionRate_%",
        "ZeroFutureSpendRate_%",
        "Insight",
        "Recommendation"
    ]
].copy()

segment_cards_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "segment_cards.csv"
    ),
    index=False
)

segment_cards_df.to_json(
    os.path.join(
        FRONTEND_DATA_DIR,
        "segment_cards.json"
    ),
    orient="records",
    indent=2
)


# SAVE CLEAN FRONTEND TABLES
profile_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "cluster_profile_frontend.csv"
    ),
    index=False
)

model_comparison_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "clustering_model_comparison_frontend.csv"
    ),
    index=False
)

kmeans_k_search_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "kmeans_k_search_frontend.csv"
    ),
    index=False
)

dbscan_search_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "dbscan_search_frontend.csv"
    ),
    index=False
)

pca_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "pca_cluster_coordinates_frontend.csv"
    ),
    index=False
)


# OPTIONAL SMALLER PCA SAMPLE FOR FRONTEND PERFORMANCE
pca_sample_df = pca_df.copy()

if len(pca_sample_df) > 1500:

    pca_sample_df = pca_sample_df.sample(
        n=1500,
        random_state=42
    )

pca_sample_df.to_csv(
    os.path.join(
        FRONTEND_DATA_DIR,
        "pca_cluster_coordinates_sampled_frontend.csv"
    ),
    index=False
)

pca_sample_df.to_json(
    os.path.join(
        FRONTEND_DATA_DIR,
        "pca_cluster_coordinates_sampled_frontend.json"
    ),
    orient="records",
    indent=2
)


# PLOT 1: KMEANS ELBOW CURVE
plt.figure(figsize=(8, 5))

plt.plot(
    kmeans_k_search_df["K"],
    kmeans_k_search_df["Inertia"],
    marker="o"
)

plt.title("KMeans Elbow Curve")

plt.xlabel("Number of Clusters")

plt.ylabel("Inertia")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "01_kmeans_elbow_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 2: KMEANS SILHOUETTE BY K
plt.figure(figsize=(8, 5))

plt.plot(
    kmeans_k_search_df["K"],
    kmeans_k_search_df["Silhouette_Score"],
    marker="o"
)

plt.title("KMeans Silhouette Score by Cluster Count")

plt.xlabel("Number of Clusters")

plt.ylabel("Silhouette Score")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "02_kmeans_silhouette_by_k.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 3: MODEL SILHOUETTE COMPARISON
comparison_plot_df = model_comparison_df.copy()

comparison_plot_df = comparison_plot_df.sort_values(
    by="Silhouette_Score",
    ascending=True
)

plt.figure(figsize=(8, 5))

plt.barh(
    comparison_plot_df["Model"],
    comparison_plot_df["Silhouette_Score"]
)

plt.title("Clustering Model Comparison")

plt.xlabel("Silhouette Score")

plt.ylabel("Model")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "03_clustering_model_silhouette_comparison.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 4: MODEL NOISE COMPARISON
noise_plot_df = model_comparison_df.copy()

noise_plot_df = noise_plot_df.sort_values(
    by="Noise_%",
    ascending=True
)

plt.figure(figsize=(8, 5))

plt.barh(
    noise_plot_df["Model"],
    noise_plot_df["Noise_%"]
)

plt.title("Noise Percentage by Clustering Model")

plt.xlabel("Noise (%)")

plt.ylabel("Model")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "04_clustering_model_noise_comparison.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 5: PCA SCATTER FOR KMEANS
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    pca_df["PCA1"],
    pca_df["PCA2"],
    c=pca_df["KMeans_Cluster"],
    alpha=0.7
)

plt.title("KMeans Customer Segments in PCA Space")

plt.xlabel("PCA Component 1")

plt.ylabel("PCA Component 2")

plt.colorbar(
    scatter,
    label="KMeans Cluster"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "05_kmeans_pca_segments.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 6: CUSTOMER SHARE BY SEGMENT
customer_share_df = profile_df.sort_values(
    by="CustomerShare_%",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    customer_share_df["SegmentName"],
    customer_share_df["CustomerShare_%"]
)

plt.title("Customer Share by Segment")

plt.xlabel("Customer Share (%)")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "06_customer_share_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 7: FUTURE REVENUE SHARE BY SEGMENT
revenue_share_df = profile_df.sort_values(
    by="FutureRevenueShare_%",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    revenue_share_df["SegmentName"],
    revenue_share_df["FutureRevenueShare_%"]
)

plt.title("Future Revenue Share by Segment")

plt.xlabel("Future Revenue Share (%)")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "07_future_revenue_share_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 8: CUSTOMER SHARE VS REVENUE SHARE
x = np.arange(
    len(profile_df)
)

width = 0.35

plt.figure(figsize=(11, 6))

plt.bar(
    x - width / 2,
    profile_df["CustomerShare_%"],
    width,
    label="Customer Share"
)

plt.bar(
    x + width / 2,
    profile_df["FutureRevenueShare_%"],
    width,
    label="Future Revenue Share"
)

plt.title("Customer Share vs Future Revenue Share")

plt.xlabel("Segment")

plt.ylabel("Share (%)")

plt.xticks(
    x,
    profile_df["SegmentName"],
    rotation=25,
    ha="right"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "08_customer_share_vs_revenue_share.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 9: RETENTION RATE VS ZERO-SPEND RATE
plt.figure(figsize=(11, 6))

plt.bar(
    x - width / 2,
    profile_df["RetentionRate_%"],
    width,
    label="Retention Rate"
)

plt.bar(
    x + width / 2,
    profile_df["ZeroFutureSpendRate_%"],
    width,
    label="Zero Future Spend Rate"
)

plt.title("Retention vs Zero-Spend Rate by Segment")

plt.xlabel("Segment")

plt.ylabel("Rate (%)")

plt.xticks(
    x,
    profile_df["SegmentName"],
    rotation=25,
    ha="right"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "09_retention_vs_zero_spend_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 10: AVERAGE FUTURE SPEND BY SEGMENT
avg_future_df = profile_df.sort_values(
    by="AvgFutureSpend90Days",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    avg_future_df["SegmentName"],
    avg_future_df["AvgFutureSpend90Days"]
)

plt.title("Average Future Spend by Segment")

plt.xlabel("Average Future Spend in Next 90 Days")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "10_avg_future_spend_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 11: AVERAGE RECENCY BY SEGMENT
recency_df = profile_df.sort_values(
    by="AvgRecency",
    ascending=False
)

plt.figure(figsize=(10, 6))

plt.barh(
    recency_df["SegmentName"],
    recency_df["AvgRecency"]
)

plt.title("Average Recency by Segment")

plt.xlabel("Average Recency in Days")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "11_avg_recency_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 12: AVERAGE FREQUENCY BY SEGMENT
frequency_df = profile_df.sort_values(
    by="AvgFrequency",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    frequency_df["SegmentName"],
    frequency_df["AvgFrequency"]
)

plt.title("Average Purchase Frequency by Segment")

plt.xlabel("Average Frequency")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "12_avg_frequency_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 13: AVERAGE MONETARY VALUE BY SEGMENT
monetary_df = profile_df.sort_values(
    by="AvgMonetary",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    monetary_df["SegmentName"],
    monetary_df["AvgMonetary"]
)

plt.title("Average Historical Monetary Value by Segment")

plt.xlabel("Average Historical Monetary Value")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "13_avg_monetary_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 14: HISTORICAL VS FUTURE REVENUE BY SEGMENT
plt.figure(figsize=(11, 6))

plt.bar(
    x - width / 2,
    profile_df["TotalHistoricalRevenue"],
    width,
    label="Historical Revenue"
)

plt.bar(
    x + width / 2,
    profile_df["TotalFutureRevenue"],
    width,
    label="Future Revenue"
)

plt.title("Historical vs Future Revenue by Segment")

plt.xlabel("Segment")

plt.ylabel("Revenue")

plt.xticks(
    x,
    profile_df["SegmentName"],
    rotation=25,
    ha="right"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "14_historical_vs_future_revenue_by_segment.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 15: STANDARDIZED SEGMENT PROFILE HEATMAP
heatmap_features = [
    "AvgRecency",
    "AvgFrequency",
    "AvgMonetary",
    "AvgBasketValue",
    "AvgFutureSpend90Days",
    "RetentionRate_%",
    "ZeroFutureSpendRate_%",
    "CustomerShare_%",
    "FutureRevenueShare_%"
]

heatmap_features = [
    feature
    for feature in heatmap_features
    if feature in profile_df.columns
]

heatmap_source = profile_df[
    heatmap_features
].copy()

heatmap_scaled = (
    heatmap_source -
    heatmap_source.mean()
) / heatmap_source.std(ddof=0)

plt.figure(figsize=(12, 6))

plt.imshow(
    heatmap_scaled,
    aspect="auto"
)

plt.colorbar(
    label="Standardized Value"
)

plt.xticks(
    ticks=np.arange(len(heatmap_features)),
    labels=heatmap_features,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(profile_df)),
    labels=profile_df["SegmentName"]
)

plt.title("Standardized Segment Profile Heatmap")

plt.xlabel("Feature")

plt.ylabel("Segment")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "15_segment_profile_heatmap.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# PLOT 16: DBSCAN NOISE VS SILHOUETTE DIAGNOSTIC
dbscan_valid_df = dbscan_search_df[
    dbscan_search_df["Silhouette_Score"].notna()
].copy()

plt.figure(figsize=(8, 6))

plt.scatter(
    dbscan_valid_df["Noise_%"],
    dbscan_valid_df["Silhouette_Score"],
    alpha=0.8
)

plt.title("DBSCAN Noise vs Silhouette Diagnostic")

plt.xlabel("Noise (%)")

plt.ylabel("Silhouette Score")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FRONTEND_PLOT_DIR,
        "16_dbscan_noise_vs_silhouette.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# FRONTEND CHART REGISTRY
chart_registry = {
    "clusteringStatus": "complete",
    "finalModel": "KMeans",
    "finalClusterCount": 4,
    "modelDecision": "KMeans with 4 clusters was selected because it gives balanced, interpretable customer groups with no unassigned noise customers.",
    "dbscanNote": "DBSCAN had higher silhouette in diagnostics but was not selected because it leaves a meaningful share of customers as noise.",
    "plots": {
        "kmeansElbow": "clustering_frontend_plots/01_kmeans_elbow_curve.png",
        "kmeansSilhouette": "clustering_frontend_plots/02_kmeans_silhouette_by_k.png",
        "modelSilhouetteComparison": "clustering_frontend_plots/03_clustering_model_silhouette_comparison.png",
        "modelNoiseComparison": "clustering_frontend_plots/04_clustering_model_noise_comparison.png",
        "pcaSegments": "clustering_frontend_plots/05_kmeans_pca_segments.png",
        "customerShare": "clustering_frontend_plots/06_customer_share_by_segment.png",
        "futureRevenueShare": "clustering_frontend_plots/07_future_revenue_share_by_segment.png",
        "customerVsRevenueShare": "clustering_frontend_plots/08_customer_share_vs_revenue_share.png",
        "retentionVsZeroSpend": "clustering_frontend_plots/09_retention_vs_zero_spend_by_segment.png",
        "avgFutureSpend": "clustering_frontend_plots/10_avg_future_spend_by_segment.png",
        "avgRecency": "clustering_frontend_plots/11_avg_recency_by_segment.png",
        "avgFrequency": "clustering_frontend_plots/12_avg_frequency_by_segment.png",
        "avgMonetary": "clustering_frontend_plots/13_avg_monetary_by_segment.png",
        "historicalVsFutureRevenue": "clustering_frontend_plots/14_historical_vs_future_revenue_by_segment.png",
        "segmentHeatmap": "clustering_frontend_plots/15_segment_profile_heatmap.png",
        "dbscanNoiseDiagnostic": "clustering_frontend_plots/16_dbscan_noise_vs_silhouette.png"
    },
    "data": {
        "segmentCards": "clustering_frontend_data/segment_cards.json",
        "segmentCardsCsv": "clustering_frontend_data/segment_cards.csv",
        "clusterProfile": "clustering_frontend_data/cluster_profile_frontend.csv",
        "pcaSample": "clustering_frontend_data/pca_cluster_coordinates_sampled_frontend.json",
        "pcaFullCsv": "clustering_frontend_data/pca_cluster_coordinates_frontend.csv",
        "modelComparison": "clustering_frontend_data/clustering_model_comparison_frontend.csv",
        "kmeansSearch": "clustering_frontend_data/kmeans_k_search_frontend.csv",
        "dbscanSearch": "clustering_frontend_data/dbscan_search_frontend.csv"
    }
}

with open(
    os.path.join(
        FRONTEND_DATA_DIR,
        "clustering_chart_registry.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chart_registry,
        f,
        indent=2
    )


# FINAL PRINT SUMMARY
print("\nSaved frontend clustering plots in:")

print(FRONTEND_PLOT_DIR)

print("\nSaved frontend clustering data in:")

print(FRONTEND_DATA_DIR)

print("\nImportant frontend data files:")

print(os.path.join(FRONTEND_DATA_DIR, "segment_cards.json"))

print(os.path.join(FRONTEND_DATA_DIR, "segment_cards.csv"))

print(os.path.join(FRONTEND_DATA_DIR, "clustering_chart_registry.json"))

print(os.path.join(FRONTEND_DATA_DIR, "pca_cluster_coordinates_sampled_frontend.json"))

print("\nImportant frontend plot files:")

print(os.path.join(FRONTEND_PLOT_DIR, "05_kmeans_pca_segments.png"))

print(os.path.join(FRONTEND_PLOT_DIR, "08_customer_share_vs_revenue_share.png"))

print(os.path.join(FRONTEND_PLOT_DIR, "09_retention_vs_zero_spend_by_segment.png"))

print(os.path.join(FRONTEND_PLOT_DIR, "15_segment_profile_heatmap.png"))

print("\nClustering frontend export complete.")


Saved frontend clustering plots in:
clustering_frontend_plots

Saved frontend clustering data in:
clustering_frontend_data

Important frontend data files:
clustering_frontend_data\segment_cards.json
clustering_frontend_data\segment_cards.csv
clustering_frontend_data\clustering_chart_registry.json
clustering_frontend_data\pca_cluster_coordinates_sampled_frontend.json

Important frontend plot files:
clustering_frontend_plots\05_kmeans_pca_segments.png
clustering_frontend_plots\08_customer_share_vs_revenue_share.png
clustering_frontend_plots\09_retention_vs_zero_spend_by_segment.png
clustering_frontend_plots\15_segment_profile_heatmap.png

Clustering frontend export complete.
